# Day 4 — Winter ONI forecast + hybrid impacts

1. Forecast winter ONI with the trained CNN  
2. Pull Open-Meteo winters for **cities first** (MVP tonight)  
3. Optional coarser **grid** later for denser click-anywhere interpolation  
4. Write `impacts.json` (Day 5 interpolates with IDW)

**Why cities-first:** Colab shared IPs get Open-Meteo **HTTP 429** constantly on a 300-point grid (hours of sleeping). ~22 cities usually finish; click-anywhere still works by interpolating between cities. Add a denser grid later from a home IP / overnight local run.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aarib-sami/ninonet/blob/main/enso/day4_impacts.ipynb)


## 0. Install


In [ ]:
!pip install -q xarray netCDF4 numpy pandas scikit-learn matplotlib requests torch pyarrow


## 1. Mount Drive


In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/ensocast/data")
OUT_DIR = Path("/content/drive/MyDrive/ensocast/artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert (DATA_DIR / "pacific_anom.nc").exists(), "Rerun Day 1"
assert (DATA_DIR / "oni_monthly.csv").exists(), "Rerun Day 1"
print("Data:", DATA_DIR)
print("Artifacts:", OUT_DIR)


## 2. Load SST anomalies + ONI


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

anom = xr.open_dataarray(DATA_DIR / "pacific_anom.nc")
if isinstance(anom, xr.Dataset):
    anom = anom[list(anom.data_vars)[0]]

oni_df = pd.read_csv(DATA_DIR / "oni_monthly.csv", parse_dates=["time"])

arr = anom.values.astype("float32")
times = pd.to_datetime(anom["time"].values).to_period("M").to_timestamp()

oni_series = oni_df.set_index("time")["oni"]
oni_series.index = pd.to_datetime(oni_series.index).to_period("M").to_timestamp()
oni = oni_series.reindex(times).to_numpy(dtype="float32")

missing = int(np.isnan(oni).sum())
if missing:
    print(f"Dropping {missing} month(s) with no ONI.")
    valid = ~np.isnan(oni)
    arr, times, oni = arr[valid], times[valid], oni[valid]

print("months:", len(arr), "last:", times[-1].date(), "last ONI:", float(oni[-1]))


## 3. Load CNN and forecast winter ONI


In [ ]:
import torch
import torch.nn as nn

LEAD = 6
WINDOW = 12
WINTER_LABEL = "2026-27"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class ENSOForecaster(nn.Module):
    def __init__(self, in_months=12, dropout=0.4, use_bn=True):
        super().__init__()
        layers = [nn.Conv2d(in_months, 32, 3, padding=1)]
        if use_bn:
            layers.append(nn.BatchNorm2d(32))
        layers += [nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1)]
        if use_bn:
            layers.append(nn.BatchNorm2d(64))
        layers += [
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


candidates = [
    OUT_DIR / f"enso_cnn_lead{LEAD}_tuned.pt",
    OUT_DIR / f"enso_cnn_lead{LEAD}.pt",
    OUT_DIR / "enso_cnn_lead3_tuned.pt",
    OUT_DIR / "enso_cnn_lead3.pt",
]
ckpt_path = next((p for p in candidates if p.exists()), None)
assert ckpt_path is not None, "No CNN checkpoint — rerun Day 2/3"
print("Loading", ckpt_path)

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
state = ckpt["model_state"]
use_bn = any("running_mean" in k for k in state)
dropout = float(ckpt.get("hp", {}).get("dropout", 0.4)) if isinstance(ckpt.get("hp"), dict) else 0.4

model = ENSOForecaster(dropout=dropout, use_bn=use_bn).to(device)
model.load_state_dict(state)
model.eval()
print("use_bn:", use_bn, "ckpt lead:", ckpt.get("lead"), "test_rmse:", ckpt.get("test_rmse"))


In [ ]:
x = arr[-WINDOW:].astype("float32")
mu, sd = ckpt.get("norm_mu"), ckpt.get("norm_sd")
if mu is not None and sd is not None:
    x = (x - float(mu)) / float(sd)
    print(f"applied norm mu={mu:.4f} sd={sd:.4f}")

x_t = torch.from_numpy(x[None]).to(device)
with torch.no_grad():
    forecast_oni = float(model(x_t).cpu().numpy().squeeze())

last_month = times[-1]
print(f"input ends: {last_month.date()}")
print(f"forecast ONI (winter {WINTER_LABEL}): {forecast_oni:.3f}")


## 4. Locations to fetch

Keep `FETCH_MODE = "cities"` tonight. Only switch to `"grid"` for an overnight local run.


In [ ]:
# "cities" = MVP tonight (~22 calls). "grid" = denser field (often dies on Colab 429).
FETCH_MODE = "cities"

LAT_MIN, LAT_MAX = 15.0, 65.0
LON_MIN, LON_MAX = -140.0, -50.0
STEP = 8.0  # used only for grid mode

CITIES = [
    ("Vancouver", 49.28, -123.12),
    ("Seattle", 47.61, -122.33),
    ("San Francisco", 37.77, -122.42),
    ("Los Angeles", 34.05, -118.24),
    ("Phoenix", 33.45, -112.07),
    ("Denver", 39.74, -104.99),
    ("Calgary", 51.05, -114.07),
    ("Winnipeg", 49.90, -97.14),
    ("Minneapolis", 44.98, -93.27),
    ("Chicago", 41.88, -87.63),
    ("Toronto", 43.65, -79.38),
    ("Montreal", 45.50, -73.57),
    ("Boston", 42.36, -71.06),
    ("New York", 40.71, -74.01),
    ("Washington DC", 38.91, -77.04),
    ("Atlanta", 33.75, -84.39),
    ("Miami", 25.76, -80.19),
    ("New Orleans", 29.95, -90.07),
    ("Houston", 29.76, -95.37),
    ("Dallas", 32.78, -96.80),
    ("Mexico City", 19.43, -99.13),
    ("Anchorage", 61.22, -149.90),
]

locations = []  # (loc_id, name_or_None, lat, lon)

if FETCH_MODE == "grid":
    lats = np.arange(LAT_MIN, LAT_MAX + 1e-6, STEP)
    lons = np.arange(LON_MIN, LON_MAX + 1e-6, STEP)
    for la in lats:
        for lo in lons:
            locations.append((f"g:{float(la):.2f},{float(lo):.2f}", None, float(la), float(lo)))
    print(f"grid: {len(lats)} x {len(lons)} = {len(lats) * len(lons)} points")
elif FETCH_MODE != "cities":
    raise ValueError('FETCH_MODE must be "cities" or "grid"')

for name, la, lo in CITIES:
    locations.append((f"c:{name}", name, la, lo))

print("FETCH_MODE:", FETCH_MODE)
print("total fetch targets:", len(locations))


## 5. Open-Meteo fetch (cached on Drive)


In [ ]:
import time
import requests

START, END = "1960-01-01", "2025-12-31"
ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"


def fetch_daily(lat, lon, retries=5):
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START,
        "end_date": END,
        "daily": "temperature_2m_mean,precipitation_sum,snowfall_sum",
        "timezone": "auto",
    }
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(ARCHIVE, params=params, timeout=120)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:
                wait = 180 + 60 * attempt
                print(f"429 — sleep {wait}s ...", end=" ", flush=True)
                time.sleep(wait)
                last_err = RuntimeError("HTTP 429")
                continue
            last_err = RuntimeError(f"HTTP {r.status_code}")
        except Exception as e:
            last_err = e
        time.sleep(3 * (attempt + 1))
    raise last_err


def daily_to_winters(payload):
    daily = payload["daily"]
    df = pd.DataFrame(
        {
            "date": pd.to_datetime(daily["time"]),
            "tmean": daily["temperature_2m_mean"],
            "precip": daily["precipitation_sum"],
            "snow": daily["snowfall_sum"],
        }
    )
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["winter_year"] = np.where(df["month"] == 12, df["year"] + 1, df["year"])
    winter = df[df["month"].isin([12, 1, 2])]
    g = winter.groupby("winter_year").agg(
        tmean=("tmean", "mean"),
        precip=("precip", "sum"),
        snow=("snow", "sum"),
        n_days=("tmean", "count"),
    )
    return g[g["n_days"] >= 85]


oni_by_winter = (
    oni_df.assign(time=pd.to_datetime(oni_df["time"]))
    .assign(month=lambda d: d["time"].dt.month, year=lambda d: d["time"].dt.year)
    .loc[lambda d: d["month"] == 1]
    .set_index("year")["oni"]
    .astype(float)
)
print("ONI winters:", int(oni_by_winter.index.min()), "->", int(oni_by_winter.index.max()))


In [ ]:
cache_path = OUT_DIR / "winter_cache.parquet"
if cache_path.exists():
    cache_df = pd.read_parquet(cache_path)
    print("loaded cache", cache_path, "rows", len(cache_df))
else:
    cache_df = pd.DataFrame(
        {
            "loc_id": pd.Series(dtype="string"),
            "winter_year": pd.Series(dtype="int64"),
            "tmean": pd.Series(dtype="float64"),
            "precip": pd.Series(dtype="float64"),
            "snow": pd.Series(dtype="float64"),
        }
    )

cached_ids = set(cache_df["loc_id"].dropna().astype(str).unique()) if len(cache_df) else set()
print("already cached locations:", len(cached_ids))

SLEEP_BETWEEN = 5.0
failures = []

for i, (loc_id, name, lat, lon) in enumerate(locations, 1):
    if loc_id in cached_ids:
        continue
    print(f"[{i}/{len(locations)}] fetch {loc_id} ...", end=" ", flush=True)
    try:
        winters = daily_to_winters(fetch_daily(lat, lon))
        part = winters.reset_index()
        part.insert(0, "loc_id", loc_id)
        part = part[["loc_id", "winter_year", "tmean", "precip", "snow"]]
        cache_df = pd.concat([cache_df, part], ignore_index=True)
        cache_df.to_parquet(cache_path, index=False)
        cached_ids.add(loc_id)
        print(len(winters), "winters | cached", len(cached_ids))
    except Exception as e:
        print("FAIL", e)
        failures.append((loc_id, str(e)))
        print("cooling 5 min ...")
        time.sleep(300)
    time.sleep(SLEEP_BETWEEN)

print("cache unique locs:", cache_df["loc_id"].nunique(), "failures:", len(failures))
print("Re-run this cell to retry failures; cached locs are skipped.")


## 6. Fit ONI → winter anomalies


In [ ]:
def phrase(var, anom):
    if var == "temp":
        if anom > 0.3:
            return "warmer than usual"
        if anom < -0.3:
            return "colder than usual"
        return "near-normal temperatures"
    if var == "precip":
        if anom > 10:
            return "wetter than usual"
        if anom < -10:
            return "drier than usual"
        return "near-normal precipitation"
    if anom > 5:
        return "snowier than usual"
    if anom < -5:
        return "less snow than usual"
    return "near-normal snowfall"


def confidence_tag(score):
    if score >= 0.7:
        return "strong"
    if score >= 0.55:
        return "moderate"
    return "mixed"


def fit_location(winters, forecast_oni, oni_by_winter):
    common = winters.join(oni_by_winter.rename("oni"), how="inner").dropna()
    if len(common) < 20:
        return None
    confidences = []
    out = {"n_winters": int(len(common))}
    for var, key in [("temp", "tmean"), ("precip", "precip"), ("snow", "snow")]:
        clim = float(common[key].mean())
        anom = common[key] - clim
        b, a = np.polyfit(common["oni"].to_numpy(), anom.to_numpy(), 1)
        pred = float(a + b * forecast_oni)
        enso = common[common["oni"] >= 0.5]
        if len(enso) >= 5 and abs(pred) > 1e-6:
            conf = float((np.sign(enso[key] - clim) == np.sign(pred)).mean())
        else:
            conf = 0.5
        confidences.append(conf)
        out[f"{var}_anom"] = pred
        out[f"{var}_phrase"] = phrase(var, pred)
        out[f"{var}_slope"] = float(b)
    out["confidence"] = float(np.mean(confidences))
    out["confidence_tag"] = confidence_tag(out["confidence"])
    return out


meta = {loc_id: (name, lat, lon) for loc_id, name, lat, lon in locations}
grid_impacts, city_impacts = [], []

for loc_id, group in cache_df.groupby("loc_id"):
    winters = group.set_index("winter_year")[["tmean", "precip", "snow"]]
    fitted = fit_location(winters, forecast_oni, oni_by_winter)
    if fitted is None:
        continue

    name = lat = lon = None
    loc_id = str(loc_id)
    if loc_id in meta:
        name, lat, lon = meta[loc_id]
    elif loc_id.startswith("g:") and "," in loc_id:
        la_s, lo_s = loc_id[2:].split(",")
        lat, lon = float(la_s), float(lo_s)
    elif loc_id.startswith("c:"):
        name = loc_id[2:]
        hit = next((c for c in CITIES if c[0] == name), None)
        if not hit:
            continue
        _, lat, lon = hit
    else:
        continue

    row = {"lat": lat, "lon": lon, "forecast_oni": float(forecast_oni), "winter": WINTER_LABEL, **fitted}
    if name is None:
        grid_impacts.append(row)
    else:
        city_impacts.append({"city": name, **row})

idw_points = grid_impacts if grid_impacts else [
    {k: v for k, v in c.items() if k != "city"} for c in city_impacts
]
print("grid:", len(grid_impacts), "cities:", len(city_impacts), "idw:", len(idw_points))


## 7. IDW helper (Day 5 click-anywhere)


In [ ]:
def interpolate_impact(lat, lon, grid, k=4, power=2.0):
    if not grid:
        raise ValueError("empty idw field")
    lats = np.array([g["lat"] for g in grid], float)
    lons = np.array([g["lon"] for g in grid], float)
    d = np.maximum(np.hypot(lats - lat, lons - lon), 1e-6)
    idx = np.argsort(d)[:k]
    w = 1.0 / (d[idx] ** power)
    w /= w.sum()

    def gather(key):
        return float(np.sum(w * np.array([grid[i][key] for i in idx], float)))

    temp, precip, snow, conf = gather("temp_anom"), gather("precip_anom"), gather("snow_anom"), gather("confidence")
    return {
        "lat": lat,
        "lon": lon,
        "temp_anom": temp,
        "precip_anom": precip,
        "snow_anom": snow,
        "temp_phrase": phrase("temp", temp),
        "precip_phrase": phrase("precip", precip),
        "snow_phrase": phrase("snow", snow),
        "confidence": conf,
        "confidence_tag": confidence_tag(conf),
        "winter": WINTER_LABEL,
        "forecast_oni": float(forecast_oni),
        "source": "idw",
    }


demo = interpolate_impact(43.65, -79.38, idw_points)
print("IDW @ Toronto:", {k: demo[k] for k in ["temp_anom", "precip_anom", "snow_anom", "confidence_tag"]})


## 8. Write impacts.json


In [ ]:
import json

payload = {
    "winter": WINTER_LABEL,
    "forecast_oni": float(forecast_oni),
    "model_checkpoint": ckpt_path.name,
    "model_lead": int(ckpt.get("lead", LEAD)),
    "input_end": str(last_month.date()),
    "fetch_mode": FETCH_MODE,
    "grid_meta": {
        "n_grid": len(grid_impacts),
        "n_cities": len(city_impacts),
        "n_idw_points": len(idw_points),
        "interpolation": "inverse_distance_weighting_k4",
        "note": "If grid empty, IDW uses city points.",
    },
    "disclaimer": (
        "Model forecast fed through a historical ONI–local winter relationship. "
        "Click-anywhere values are interpolated. Not an official outlook."
    ),
    "grid": grid_impacts,
    "idw_points": idw_points,
    "cities": city_impacts,
}

for path in [OUT_DIR / "impacts.json", DATA_DIR / "impacts.json"]:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("Wrote", path)

print(f"ONI={forecast_oni:.3f} cities={len(city_impacts)} idw={len(idw_points)}")
print("Day 4 checkpoint: impacts ready.")


## 9. Preview


In [ ]:
import matplotlib.pyplot as plt

plot_df = pd.DataFrame(idw_points)
fig, ax = plt.subplots(figsize=(11, 5))
sc = ax.scatter(
    plot_df["lon"], plot_df["lat"], c=plot_df["temp_anom"],
    cmap="RdBu_r", vmin=-2, vmax=2, s=90, edgecolors="k", linewidths=0.3,
)
for c in city_impacts:
    ax.annotate(c["city"], (c["lon"], c["lat"]), fontsize=7, alpha=0.85)
plt.colorbar(sc, ax=ax, label="temp anomaly (C)")
ax.set_title(f"Winter {WINTER_LABEL} | ONI={forecast_oni:.2f} | {FETCH_MODE}")
ax.set_xlim(-170, -50)
ax.set_ylim(15, 65)
plt.tight_layout()
fig.savefig(OUT_DIR / "impacts_grid_preview.png", dpi=140)
plt.show()
